# agent_loop_error_handling

目标： 让 Mini Agent Loop 在工具调用失败、参数错误、任务不存在、模型输出异常时，能够稳定处理。
- 统一工具返回格式
- 工具参数校验
- 工具调用日志
- max_tool_rounds 控制
- 错误恢复

In [ ]:
# User input
# ↓
# LLM 判断是否需要 tool
# ↓
# 尝试解析 tool_call
# ↓
# 如果解析失败：生成错误信息，回传模型
# ↓
# 如果 tool 不存在：生成错误信息，回传模型
# ↓
# 如果参数不合法：生成错误信息，回传模型
# ↓
# 如果工具执行报错：捕获异常，回传模型
# ↓
# 如果循环次数过多：停止，给用户一个安全回答
# ↓
# 模型基于成功结果或错误结果生成最终回答

## 导入依赖

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any, Callable, Literal, TypedDict, cast

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam


## 配置 DeepSeek Client

In [ ]:
load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if DEEPSEEK_API_KEY is None:
    raise RuntimeError(
        "Missing DEEPSEEK_API_KEY. Please put it in your .env file."
    )

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)

# 你可以在 .env 里写：
# DEEPSEEK_MODEL=deepseek-v4-flash
#
# 或者按你当前实际使用的最新 DeepSeek 模型改这里。
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

## 定义私有数据

In [ ]:
class PrivateNote(TypedDict):
    birthday: str
    internal_code: str
    role: str


PRIVATE_NOTES: dict[str, PrivateNote] = {
    "Jack": {
        "birthday": "3月18日",
        "internal_code": "A-102",
        "role": "backend intern",
    },
    "Alice": {
        "birthday": "7月2日",
        "internal_code": "B-207",
        "role": "product manager",
    },
    "Bob": {
        "birthday": "11月9日",
        "internal_code": "C-315",
        "role": "data engineer",
    },

    # 这个人专门用于测试 ToolRuntimeError。
    # 正常业务里不需要这种数据，这里只是 error handling test fixture。
    "CrashTest": {
        "birthday": "1月1日",
        "internal_code": "X-000",
        "role": "runtime error test user",
    },
}

## 定义统一 Tool Result 类型

In [ ]:
class SuccessResult(TypedDict):
    ok: Literal[True]
    data: dict[str, Any]


class BaseErrorResult(TypedDict):
    ok: Literal[False]
    error_type: str
    message: str


class ErrorResult(BaseErrorResult, total=False):
    available_tools: list[str]
    details: dict[str, Any]


ToolResult = SuccessResult | ErrorResult


def success_result(data: dict[str, Any]) -> SuccessResult:
    return {
        "ok": True,
        "data": data,
    }


def error_result(
    error_type: str,
    message: str,
    *,
    available_tools: list[str] | None = None,
    details: dict[str, Any] | None = None,
) -> ErrorResult:
    result: ErrorResult = {
        "ok": False,
        "error_type": error_type,
        "message": message,
    }

    if available_tools is not None:
        result["available_tools"] = available_tools

    if details is not None:
        result["details"] = details

    return result

## 定义两个极简工具

In [ ]:
def get_private_note(name: str) -> ToolResult:
    """
    Get private note by exact person name.

    This function intentionally knows information that the model does not know.
    So the model should call this tool instead of guessing.
    """
    name = name.strip()

    if not name:
        return error_result(
            error_type="ValidationError",
            message="name cannot be empty.",
        )

    if name == "CrashTest":
        raise RuntimeError("Simulated database connection failure.")

    note = PRIVATE_NOTES.get(name)

    if note is None:
        return error_result(
            error_type="NotFoundError",
            message=f"Private note for '{name}' was not found.",
            details={
                "requested_name": name,
                "known_names": list(PRIVATE_NOTES.keys()),
            },
        )

    return success_result(
        {
            "name": name,
            "birthday": note["birthday"],
            "internal_code": note["internal_code"],
            "role": note["role"],
        }
    )


def search_private_notes(keyword: str) -> ToolResult:
    """
    Search private notes by keyword.
    """
    keyword_normalized = keyword.strip().lower()

    if not keyword_normalized:
        return error_result(
            error_type="ValidationError",
            message="keyword cannot be empty.",
        )

    matches: list[dict[str, str]] = []

    for name, note in PRIVATE_NOTES.items():
        searchable_text = " ".join(
            [
                name,
                note["birthday"],
                note["internal_code"],
                note["role"],
            ]
        ).lower()

        if keyword_normalized in searchable_text:
            matches.append(
                {
                    "name": name,
                    "birthday": note["birthday"],
                    "internal_code": note["internal_code"],
                    "role": note["role"],
                }
            )

    if not matches:
        return error_result(
            error_type="NoSearchResultError",
            message=f"No private notes matched keyword: {keyword}",
            details={
                "keyword": keyword,
            },
        )

    return success_result(
        {
            "keyword": keyword,
            "matches": matches,
        }
    )


## 定义 Tool Schema

In [ ]:
TOOLS: list[ChatCompletionToolParam] = [
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "get_private_note",
                "description": (
                    "Get one private note by exact person name. "
                    "Use this when the user asks about a specific person."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "Exact person name, for example Jack.",
                        },
                    },
                    "required": ["name"],
                },
            },
        },
    ),
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "search_private_notes",
                "description": (
                    "Search private notes by keyword. "
                    "Use this when the user does not provide an exact name."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "keyword": {
                            "type": "string",
                            "description": "Keyword to search in names, birthdays, codes, or roles.",
                        },
                    },
                    "required": ["keyword"],
                },
            },
        },
    ),
]


## 建立 Function Dispatch

In [ ]:
ToolFunction = Callable[..., ToolResult]


TOOL_MAP: dict[str, ToolFunction] = {
    "get_private_note": get_private_note,
    "search_private_notes": search_private_notes,
}


AVAILABLE_TOOL_NAMES = sorted(TOOL_MAP)


## 定义 Agent 的 System Prompt 和 Messages

In [ ]:
SYSTEM_PROMPT = """
You are a Mini Private Note Agent.

Your job is to answer questions about private notes by using tools.

You have access to these tools:
- get_private_note
- search_private_notes

Rules:
1. If the user asks about private note data, use a tool. Do not guess private data.
2. Tool results always use this shape: {"ok": true, "data": ...} or {"ok": false, "error_type": ..., "message": ...}.
3. If a tool returns ok=false, explain the failure clearly in Chinese and suggest a concrete next step.
4. If a tool succeeds, answer from the returned data only.
5. Keep the final answer concise and practical.
""".strip()


messages: list[ChatCompletionMessageParam] = [
    cast(
        ChatCompletionMessageParam,
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
    )
]


def reset_agent_messages() -> None:
    """
    Reset conversation messages.
    """
    global messages

    messages = [
        cast(
            ChatCompletionMessageParam,
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
        )
    ]


## Tool Call 日志与测试用假对象

In [ ]:
class ToolExecution(TypedDict):
    tool_call_id: str
    tool_name: str
    raw_arguments: str
    arguments: dict[str, Any]
    result: ToolResult


TOOL_EXECUTION_LOG: list[ToolExecution] = []


def reset_tool_execution_log() -> None:
    TOOL_EXECUTION_LOG.clear()


@dataclass
class FakeFunctionCall:
    name: str
    arguments: str


@dataclass
class FakeToolCall:
    id: str
    function: FakeFunctionCall | None
    type: str = "function"


## Tool Call 解析与执行

In [ ]:
def parse_tool_arguments(raw_arguments: str | None) -> dict[str, Any]:
    """
    Parse tool call arguments from JSON string.

    模型返回的 function.arguments 通常是 JSON string。
    这里专门把解析错误变成 ValueError，交给 execute_tool_call 统一包装。
    """
    if raw_arguments is None or raw_arguments.strip() == "":
        return {}

    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON arguments: {raw_arguments}") from error

    if not isinstance(parsed, dict):
        raise ValueError(
            f"Tool arguments must be a JSON object, got: {type(parsed).__name__}"
        )

    return cast(dict[str, Any], parsed)


def execute_tool_call(tool_call: Any) -> ToolExecution:
    """
    Execute one tool call returned by the model.

    所有失败都被转换为统一 ToolResult，避免 agent loop 因异常中断。
    """
    tool_call_id = str(getattr(tool_call, "id", ""))
    function_obj = getattr(tool_call, "function", None)

    if function_obj is None:
        execution: ToolExecution = {
            "tool_call_id": tool_call_id,
            "tool_name": "",
            "raw_arguments": "",
            "arguments": {},
            "result": error_result(
                error_type="MissingFunctionObjectError",
                message="This tool call does not contain a function object.",
                available_tools=AVAILABLE_TOOL_NAMES,
            ),
        }
        TOOL_EXECUTION_LOG.append(execution)
        return execution

    tool_name = str(getattr(function_obj, "name", ""))
    raw_arguments = str(getattr(function_obj, "arguments", "{}"))

    try:
        arguments = parse_tool_arguments(raw_arguments)
    except ValueError as error:
        execution = {
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "raw_arguments": raw_arguments,
            "arguments": {},
            "result": error_result(
                error_type="InvalidToolArgumentsError",
                message=str(error),
                details={"raw_arguments": raw_arguments},
            ),
        }
        TOOL_EXECUTION_LOG.append(execution)
        return execution

    tool_function = TOOL_MAP.get(tool_name)

    if tool_function is None:
        execution = {
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "raw_arguments": raw_arguments,
            "arguments": arguments,
            "result": error_result(
                error_type="UnknownToolError",
                message=f"Unknown tool: {tool_name}",
                available_tools=AVAILABLE_TOOL_NAMES,
            ),
        }
        TOOL_EXECUTION_LOG.append(execution)
        return execution

    try:
        result = tool_function(**arguments)
    except TypeError as error:
        result = error_result(
            error_type="ToolArgumentMismatchError",
            message=f"Tool argument mismatch for {tool_name}: {error}",
            details={
                "tool_name": tool_name,
                "arguments": arguments,
            },
        )
    except Exception as error:
        result = error_result(
            error_type="ToolRuntimeError",
            message=f"Unexpected error while running {tool_name}: {error}",
            details={
                "tool_name": tool_name,
                "exception_type": type(error).__name__,
            },
        )

    execution = {
        "tool_call_id": tool_call_id,
        "tool_name": tool_name,
        "raw_arguments": raw_arguments,
        "arguments": arguments,
        "result": result,
    }
    TOOL_EXECUTION_LOG.append(execution)
    return execution


## 把 Assistant / Tool Message 转成可再次传入 API 的格式

In [ ]:
def assistant_message_to_param(message: Any) -> ChatCompletionMessageParam:
    """
    Convert SDK response message object to ChatCompletionMessageParam.
    """
    message_param: dict[str, Any] = {
        "role": "assistant",
    }

    content = getattr(message, "content", None)
    if content is not None:
        message_param["content"] = content

    tool_calls = getattr(message, "tool_calls", None)

    if tool_calls:
        converted_tool_calls: list[dict[str, Any]] = []

        for tool_call in tool_calls:
            function_obj = getattr(tool_call, "function", None)

            if function_obj is None:
                converted_tool_calls.append(
                    {
                        "id": getattr(tool_call, "id", ""),
                        "type": getattr(tool_call, "type", "function"),
                        "function": {
                            "name": "",
                            "arguments": "{}",
                        },
                    }
                )
                continue

            converted_tool_calls.append(
                {
                    "id": getattr(tool_call, "id", ""),
                    "type": getattr(tool_call, "type", "function"),
                    "function": {
                        "name": getattr(function_obj, "name", ""),
                        "arguments": getattr(function_obj, "arguments", "{}"),
                    },
                }
            )

        message_param["tool_calls"] = converted_tool_calls

    return cast(ChatCompletionMessageParam, message_param)


def tool_result_to_message(
    tool_call_id: str,
    result: ToolResult,
) -> ChatCompletionMessageParam:
    """
    Convert local Python tool result to tool message.
    """
    return cast(
        ChatCompletionMessageParam,
        {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": json.dumps(result, ensure_ascii=False),
        },
    )


## 调用模型

In [ ]:
def call_llm() -> Any:
    """
    Call LLM once with current messages and tool schemas.
    """
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )

    return completion.choices[0].message


## 实现带错误处理的 Mini Agent Loop

In [ ]:
def print_tool_execution(execution: ToolExecution) -> None:
    """
    Pretty print one tool execution for notebook debugging.
    """
    print("\n[Tool Call]")
    print("tool_call_id:", execution["tool_call_id"])
    print("tool_name:", execution["tool_name"])
    print("raw_arguments:", execution["raw_arguments"])
    print(
        "arguments:",
        json.dumps(
            execution["arguments"],
            ensure_ascii=False,
            indent=2,
        ),
    )
    print(
        "result:",
        json.dumps(
            execution["result"],
            ensure_ascii=False,
            indent=2,
        ),
    )


def run_agent(
    user_input: str,
    *,
    verbose: bool = True,
    max_tool_rounds: int = 3,
) -> str:
    """
    Run a mini agent loop with error handling.

    处理的错误类型：
    - tool_call JSON 解析失败
    - tool 不存在
    - tool 参数不匹配
    - tool 执行时抛异常
    - 模型调用失败
    - 模型持续调用工具导致超过 max_tool_rounds
    """
    user_message = cast(
        ChatCompletionMessageParam,
        {
            "role": "user",
            "content": user_input,
        },
    )
    messages.append(user_message)

    for round_index in range(max_tool_rounds + 1):
        try:
            assistant_message = call_llm()
        except Exception as error:
            failure = error_result(
                error_type="LLMCallError",
                message=f"Model call failed: {error}",
                details={"exception_type": type(error).__name__},
            )
            if verbose:
                print(json.dumps(failure, ensure_ascii=False, indent=2))
            return "模型调用失败，Agent Loop 已安全停止。请检查网络、API Key 或模型名称。"

        messages.append(assistant_message_to_param(assistant_message))

        tool_calls = getattr(assistant_message, "tool_calls", None)

        if not tool_calls:
            final_answer = getattr(assistant_message, "content", None)
            if final_answer is None or str(final_answer).strip() == "":
                return "模型没有返回可用内容，Agent Loop 已安全停止。"

            return str(final_answer)

        if round_index >= max_tool_rounds:
            return (
                "工具调用轮数超过限制，Agent Loop 已停止。"
                "请检查模型是否陷入重复调用工具，或调大 max_tool_rounds。"
            )

        for tool_call in tool_calls:
            execution = execute_tool_call(tool_call)

            if verbose:
                print_tool_execution(execution)

            messages.append(
                tool_result_to_message(
                    tool_call_id=execution["tool_call_id"],
                    result=execution["result"],
                )
            )

    return "Agent Loop ended unexpectedly."


## 本地测试：工具成功

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_success",
        function=FakeFunctionCall(
            name="get_private_note",
            arguments=json.dumps({"name": "Jack"}, ensure_ascii=False),
        ),
    )
)
print_tool_execution(execution)


## 本地测试：业务错误 NotFound

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_not_found",
        function=FakeFunctionCall(
            name="get_private_note",
            arguments=json.dumps({"name": "Tom"}, ensure_ascii=False),
        ),
    )
)
print_tool_execution(execution)


## 本地测试：参数 JSON 解析失败

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_bad_json",
        function=FakeFunctionCall(
            name="get_private_note",
            arguments="{name: Jack}",
        ),
    )
)
print_tool_execution(execution)


## 本地测试：参数不匹配

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_argument_mismatch",
        function=FakeFunctionCall(
            name="get_private_note",
            arguments=json.dumps({"person": "Jack"}, ensure_ascii=False),
        ),
    )
)
print_tool_execution(execution)


## 本地测试：未知工具

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_unknown_tool",
        function=FakeFunctionCall(
            name="delete_private_note",
            arguments=json.dumps({"name": "Jack"}, ensure_ascii=False),
        ),
    )
)
print_tool_execution(execution)


## 本地测试：工具运行时异常

In [ ]:
execution = execute_tool_call(
    FakeToolCall(
        id="call_runtime_error",
        function=FakeFunctionCall(
            name="get_private_note",
            arguments=json.dumps({"name": "CrashTest"}, ensure_ascii=False),
        ),
    )
)
print_tool_execution(execution)


## 端到端测试：正常查询

In [ ]:
reset_agent_messages()
reset_tool_execution_log()

answer = run_agent(
    "Jack 的生日是什么？",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


## 端到端测试：搜索查询

In [ ]:
answer = run_agent(
    "帮我找一下 role 里面和 data 有关的人。",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


## 端到端测试：查不到数据时的恢复

In [ ]:
answer = run_agent(
    "Tom 的内部代码是多少？",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


## 查看工具调用日志

In [ ]:
for index, execution in enumerate(TOOL_EXECUTION_LOG, start=1):
    print(f"\n--- Tool Execution {index} ---")
    print(json.dumps(execution, ensure_ascii=False, indent=2))


## 查看完整 messages

In [ ]:
for index, message in enumerate(messages):
    print(f"\n--- Message {index} ---")
    print(json.dumps(message, ensure_ascii=False, indent=2))


## END